# A big earthquake has just struck. What will the next week bring?

**EPS 88 · PyEarth.** Open your own copy on DataHub: [click here](https://datahub.berkeley.edu/hub/user-redirect/git-pull?repo=https%3A//github.com/AI4EPS/EPS88_PyEarth&branch=main&urlpath=lab/tree/EPS88_PyEarth/docs/notebooks/05_what_comes_next.ipynb).

On the morning of 4 July 2019 a magnitude 6.4 earthquake shook the desert near the town of Ridgecrest, California. For a day and a half it looked like the main event. Then, 34 hours later, a magnitude 7.1 struck close by, the largest earthquake in California in twenty years.

After a large earthquake everybody asks the same thing: is it over, or is another one coming? Nobody can predict the next earthquake. What seismologists can give is a **forecast**, like a weather forecast: how many earthquakes to expect in the coming days, and the probability that one of them is large.

**Today you make that forecast yourself**, for the week that began ten days after the magnitude 7.1, and then check it against what happened.

Every place you write something opens with a pencil icon and the words *Your turn*, and is
followed by a cell that already has the shape of the answer written in, with `...` wherever your
code goes. Replace every `...`, run every cell so your answers and figures are
saved in the file, then **download this notebook itself — the `.ipynb` file — and upload it
to Gradescope.** Not a PDF: the marking reads your notebook, and a PDF cannot be read.
In JupyterLab: **File ▸ Download**, or right-click the file in the left-hand panel and choose
**Download**.

Two habits from the first minute. A cell runs when you press **Shift+Enter**, and the notebook
remembers everything it has already run — so when something breaks and you cannot see why,
**Kernel → Restart Kernel and Run All Cells** throws the memory away and rebuilds it from the
top. That is never the wrong thing to do.

## What you'll be able to do

**The science.** Say what a large earthquake does to the earthquakes around it, how fast its
aftershocks die away, and how a forecast for the coming week is made and read.

**The statistics.** A rate · a probability as a proportion · an expected number · a simulation
and the distribution it gives · the mean and the standard deviation · the three rules for "not",
"and" and "at least one".

**The code.** A proportion in one line, `(condition).mean()` · a random draw you can repeat,
`np.random.default_rng` and `rng.choice` · a loop that lives through the same week 2,000 times ·
`.mean()` and `.std()` of the results.

**Eight places where you write something: five in class, three at home.** Each is headed
*Your turn*, and the cell under it already has the shape of the answer written in, with `...`
wherever your code goes. Replace every `...`; a comment beside each one says what it is for. A
`...` left in place is not code, and the cell will not do what you expect until it is replaced.

1. What did the mainshock change?
2. How many aftershocks will the next week bring?
3. How big will they be?
4. What is the probability of a magnitude 4 or more next week?

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# house style, set once, so every plot cell below holds only what matters
plt.rcParams.update({"figure.dpi": 110,
                     "axes.grid": True, "grid.alpha": 0.3, "axes.axisbelow": True})

CACHE = "https://raw.githubusercontent.com/AI4EPS/EPS88_PyEarth/main/data"

def load(name, start, end, region):
    """Fetch one window of the USGS catalogue; fall back to the copy stored with the course.

    `region` is extra query text limiting the search to one area.
    """
    # Ask the live archive first. If it is down, or you are offline, read the copy stored with
    # the course instead, so the notebook still runs.
    try:
        return pd.read_csv(f"https://earthquake.usgs.gov/fdsnws/event/1/query?format=csv&orderby=time-asc"
                           f"&starttime={start}&endtime={end}&minmagnitude=2.5"
                           + region)
    except Exception as e:
        print("live source unreachable, using the cached copy:", type(e).__name__)
        return pd.read_csv(CACHE + "/" + f"week05_{name}_{start}_{end}_M2.5.csv")

# Every earthquake of magnitude 2.5 or more around two California earthquakes: Ridgecrest in 2019
# for class, Loma Prieta in 1989 for the homework.
quakes = load("ridgecrest", "2019-01-01", "2020-07-01",
              "&minlatitude=35.2&maxlatitude=36.4&minlongitude=-118.2&maxlongitude=-117.0")
loma = load("lomaprieta", "1989-01-01", "1990-10-18",
            "&latitude=37.04&longitude=-121.88&maxradiuskm=60")

# One new column in each table, `days`: the time since its largest earthquake, the mainshock.
# Negative means before it. This uses dates, which you are not expected to follow yet.
for table in [quakes, loma]:
    times = pd.to_datetime(table["time"])
    mainshock = times[table["mag"].idxmax()]
    table["days"] = (times - mainshock).dt.total_seconds() / 86400

pd.set_option("display.width", 100)     # room for a table of earthquakes to print on one line
print("Ridgecrest:", quakes.shape, "   Loma Prieta:", loma.shape)

## 1. What did the mainshock change?

**An earthquake** is a sudden slip on a fault, a break in the rock of the Earth's crust. Its
size is the **magnitude**: each step of 1 is about 32 times the energy, as in the plotting week.

**The data.** `quakes` is a piece of the public earthquake catalogue of the US Geological Survey
(USGS): every event of magnitude 2.5 or more in a box about 120 km across around Ridgecrest,
from the start of 2019 to the middle of 2020. A magnitude 2.5 is about the smallest that
people nearby can feel. One row is one event (all but two are earthquakes; the two are quarry
blasts). Today you need four columns:

- `mag`: the magnitude;
- `latitude` and `longitude`: the place;
- `days`: the time since the magnitude 7.1, in days. Negative means before it; 1.5
  means a day and a half after it.

The plot shows every row: time across, magnitude up.

In [ ]:
plt.figure(figsize=(7, 3.6))
plt.scatter(quakes["days"], quakes["mag"], s=4)
plt.xlim(-190, 60)
plt.xlabel("days since the magnitude 7.1")
plt.ylabel("magnitude")
plt.title(f"Ridgecrest, six months before to two months after (n = {len(quakes)} in all)")
plt.show()

print(quakes[quakes["mag"] >= 6][["days", "mag", "place"]])       # the two largest

For six months there is a dot every week or so: the ordinary activity of the region, called the
**background**. Then come the magnitude 6.4, the magnitude 7.1
34 hours after it, and a cloud of earthquakes that thins out as the days pass.
Together they are a **sequence**, and its pieces have names:

- the **mainshock** is the largest earthquake of the sequence, here the magnitude 7.1;
- an **aftershock** is one of the earthquakes that follow it. A large earthquake changes the stress in the rock around it, and parts of the same fault and faults near it that were already close to failing then fail too — most within hours and days, fewer and fewer over the months that follow.
- a **foreshock** is an earthquake followed by a larger one nearby. The magnitude 6.4
  was one, and nobody could tell until the larger one came.

### ✏️ Your turn 1

**How many earthquakes a day, before the sequence and after the mainshock?**

**Why:** the two numbers together say how much the mainshock changed the region.

**A rate is a count divided by the time it took**: 30 earthquakes in 10 days is a rate of 3 a
day. A rate lets you compare periods of different length.

Work out the rate for two groups of rows. Pick each group with a mask; `len` counts its rows.

- `before`: the quiet period, more than 2 days before the mainshock. It is 184 days
  long. (The cut at 2 days keeps the foreshock out of it.)
- `month`: the first 30 days after the mainshock.

**Use these names**, because the self-check looks for them: `before`, `month`, `rate_before`,
`rate_month`.

In [ ]:
# ← your answer here: replace every ... with code
before = quakes[...]           # the rows more than 2 days before the mainshock: days below -2
month = quakes[(...) & (...)]  # the first 30 days after it: days above 0, and days below 30
rate_before = len(...) / 184   # earthquakes a day, over the 184 days before
rate_month = len(...) / 30     # earthquakes a day, over the first 30 days

print("before:", len(before), "earthquakes in 184 days,", round(rate_before, 2), "a day")
print("first month:", len(month), "earthquakes in 30 days,", round(rate_month), "a day")
print("the rate went up", round(rate_month / rate_before), "times")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert (before["days"] < -2).all() and len(before) == (quakes["days"] < -2).sum(), "before keeps every row whose days is below -2, and no other"
assert (month["days"] > 0).all() and (month["days"] < 30).all() and len(month) == ((quakes["days"] > 0) & (quakes["days"] < 30)).sum(), "month keeps every row with days above 0 and below 30, and no other: two conditions joined with &"
assert abs(rate_before - len(before) / 184) < 1e-9 and abs(rate_month - len(month) / 30) < 1e-9, "each rate is the number of rows, len(...), divided by the number of days"
print(f"✓ Your turn 1 — {round(rate_before, 2)} earthquakes a day before, {round(rate_month)} a day in the first month")

**One earthquake a week became 73 a day**: the rate went up more than five
hundred times. The map shows where they were: your `before` in grey, everything after the
mainshock in red.

In [ ]:
after = quakes[quakes["days"] > 0]                # every row after the mainshock

plt.figure(figsize=(6, 5.4))
plt.scatter(after["longitude"], after["latitude"], s=2, color="crimson", label="after the mainshock")
plt.scatter(before["longitude"], before["latitude"], s=14, color="0.35", label="in the six months before")
plt.scatter(-117.6709, 35.6225, s=60, marker="s", color="black")
plt.text(-117.6709 - 0.33, 35.6225 - 0.02, "Ridgecrest")
plt.gca().set_aspect("equal")
plt.xlabel("longitude (degrees east)")
plt.ylabel("latitude (degrees north)")
plt.title(f"Before (n = {len(before)}) and after (n = {len(after)})")
plt.legend(loc="lower left")
plt.show()

**Aftershocks happen on and around the fault that slipped in the mainshock.** Most of the red
dots draw one band about 50 km long, running from north-west to south-east past the town: that
is the fault.

The rate jumped, and since then it has been falling. To forecast the days ahead you need to know
how fast it falls.

## 2. How many aftershocks will the next week bring?

**Put yourself ten days after the mainshock.** You know everything up to now and nothing after.
From here on, every earthquake in the box counts as an aftershock: at
73 a day against one a week, nearly all of them are.

**First count the aftershocks on each of the ten days.** Day 1 is the first 24 hours after the
mainshock, day 2 the next 24 hours, and so on. The `days` column is the time since the
mainshock, so:

| day number | `days` column |
|---|---|
| day 1 | from 0 to 1 |
| day 2 | from 1 to 2 |
| day 10 | from 9 to 10 |
| days 11 to 17, the week to forecast | from 10 to 17 |

`plt.hist` draws the count of each day as a bar; `np.histogram` with the same bins gives the
counts as numbers.

In [ ]:
first_ten = quakes[(quakes["days"] > 0) & (quakes["days"] < 10)]   # the first 10 days after the mainshock

plt.figure(figsize=(7, 3.6))
plt.hist(first_ten["days"], bins=np.arange(0, 11))           # one bin for each day
plt.xlabel("days since the mainshock")
plt.ylabel("aftershocks in the day")
plt.title(f"The first ten days (n = {len(first_ten)} aftershocks)")
plt.show()

daily, edges = np.histogram(first_ten["days"], bins=np.arange(0, 11))   # the same counts, as numbers
print("aftershocks on days 1 to 10:", daily)

**The count falls fast at first, then more and more slowly**: 1257 on day 1,
170 on day 2, 76 on day 4, 29 on day 8. From day 2 on, each
time the day number doubles, the count roughly halves.

**Omori's law.** The number of aftershocks in a day falls roughly as one over the time since the mainshock: twice as late, about half as many. Fusakichi Omori found it in Japan in 1894. As a formula:

***count on a day = K / day***

- **K is one number that belongs to the sequence.** A larger K means more aftershocks on every
  day.
- **A formula that stands in for the data is called a model.** Once you know K, the model is a
  forecast: put in a day that has not happened yet.

### ✏️ Your turn 2

**Which K fits this sequence, and how many aftershocks should the next seven days bring?**

**Why:** the model can forecast only when its K is known, and K is found from the days already
counted.

**Step 1: find K.** The figure shows the count of each past day as a dot, and the model,
`K / past_days`, as a red curve. Start with `K = 100`, run the cell, and change K until the curve
passes through the dots from day 2 on. The dot of day 1 stays far above the curve: in the first
hours the count falls too fast for so simple a formula.

**Step 2: forecast.** Use the same model on the coming days, days 11 to
17. `rates` is the count the model expects on each of them, and `expected` is
their sum.

**Use these names**, because the self-check looks for them: `K`, `rates`, `expected`.

In [ ]:
# ← your answer here: replace every ... with code
# Step 1: find K
past_days = np.arange(1, 11)  # the day numbers 1 to 10; daily holds the count of each
K = ...                       # try 100 first, then change it until the curve passes through the dots

plt.figure(figsize=(7, 3.6))
plt.scatter(past_days, daily, label="the count of each day")
plt.plot(past_days, K / past_days, color="firebrick", label="the model, K / day")
plt.yscale("log")  # a log axis, so that 1,000 and 30 can both be read
plt.xlabel("day after the mainshock")
plt.ylabel("aftershocks in the day")
plt.title(f"Omori's law with K = {K} (n = {len(past_days)} days)")
plt.legend()
plt.show()

# Step 2: forecast
coming_days = np.arange(11, 18)  # the day numbers 11 to 17: the week to forecast
rates = ... / ...                # the model on the coming days: K over coming_days
expected = (...).sum()           # the seven rates added up

print("K =", K)
print("expected on each of days 11 to 17:", rates.round())
print("expected in days 11 to 17:", round(expected, 1))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert 250 <= K <= 400, "choose K so that the curve passes through the dots from day 2 on: a value between about 250 and 400"
assert len(rates) == 7 and abs(rates[0] - K / coming_days[0]) < 1e-9, "rates is the model on the COMING days, K / coming_days: seven expected counts, for days 11 to 17"
assert abs(expected - rates.sum()) < 1e-9, "expected is the seven rates added up"
print(f"✓ Your turn 2 — with K = {K}, about {round(expected)} aftershocks are expected in days 11 to 17")

**A K near 320 takes the curve through the dots from day 2 on.** With it the model expects
29 aftershocks on day 11, falling to 19 on day
17: **163 in the week**. Your K may differ a little, and so may your total.
So that everyone's numbers agree, the notebook uses 163 from here on and calls it `n_week`.

## 3. How big will they be?

About 163 aftershocks are coming next week. Most will be small. **What is the probability
that an aftershock is large?**

### What a probability is

**Probability.** The proportion of many tries in which a thing happens. It is a number between 0 and 1: 0 means never, 1 means
always.

- A coin lands heads with probability 0.5: in many tosses, about half are heads.
- A die shows a six with probability 1/6, about 0.17: in many rolls, about one in six is a six.
- A forecast of "30% rain" says: of many days like this one, it rains on about 3 in 10.

**A probability does not say what the next try will be.** The next roll is a six or it is not.
It says what happens over many tries.

**Where does the number come from?** For a die you can reason it out: six faces, all alike, so
1/6. For an aftershock nobody can. So you **measure** it: take many aftershocks that have already
happened, and find the proportion of them that were large.

### Measuring a probability from the data

**The tries are the aftershocks already recorded.** `sample` holds the aftershocks from 1 day to
10 days after the mainshock, which is `days` from 1 to 10:

- **nothing after 10 days**, because that is now: a forecast can use only what has happened;
- **not the first 24 hours**, because the catalogue of that day is not complete. Right after a large
  earthquake the waves of many aftershocks overlap, and only the larger ones can be picked out.
  Counting that day would make large aftershocks look more common than they are.

The cell draws the magnitudes of the sample. Then it works out one proportion step by step, on
the first ten of them.

In [ ]:
sample = quakes[(quakes["days"] >= 1) & (quakes["days"] < 10)]   # from 1 day to 10 days after the mainshock: the first 24 hours are left out

plt.figure(figsize=(9, 3.6))
plt.subplot(1, 2, 1)
plt.hist(sample["mag"], bins=np.arange(2.5, 5.2, 0.1))
plt.axvline(4, color="firebrick")
plt.xlabel("magnitude")
plt.ylabel("number of aftershocks")
plt.title("ordinary axis")

plt.subplot(1, 2, 2)
plt.hist(sample["mag"], bins=np.arange(2.5, 5.2, 0.1))
plt.axvline(4, color="firebrick")
plt.yscale("log")
plt.xlabel("magnitude")
plt.title(f"log axis (n = {len(sample)} aftershocks)")
plt.show()

ten = np.array(sample["mag"])[:10]                # the first ten magnitudes of the sample
print("ten magnitudes:    ", ten)
print("3 or more?         ", ten >= 3)
print("how many are True: ", (ten >= 3).sum(), "of 10")
print("the proportion:    ", (ten >= 3).mean())

**Small aftershocks far outnumber large ones.** The bars fall away quickly to the right. The
red line marks magnitude 4, large enough to be felt well beyond the place where it happens. The
few bars to its right show only on the log axis.

**A proportion is the mean of a mask.** `ten >= 3` asks "is this one magnitude 3 or more?" of
every value and answers True or False. 5 of the ten are True: a proportion of
0.5. Python counts True as 1 and False as 0, so `.mean()` of the mask gives the
proportion in one step.

**The proportion in the whole sample is the probability to use for next week.** Ten values are
too few to trust; the sample has 637. This assumes one thing: the sizes of next week's
aftershocks will be like the sizes in the sample.

**Expected number.** How many to expect on average: the number of tries times the probability of each. With 10 tries and a probability of 0.5, expect
10 × 0.5 = 5. It is an average, so it does not have to be a whole number.

### ✏️ Your turn 3

**What is the probability that an aftershock is magnitude 3 or more, and magnitude 4 or more? How
many of each should next week bring?**

**Why:** the proportion in the sample is the probability for one aftershock, and with it you can
say how many large aftershocks to expect among next week's.

1. Two proportions, on the whole `sample`, as the cell above did on ten values.
2. Two expected numbers. The tries are next week's aftershocks, `n_week` of them.

**Use these names**, because the self-check looks for them: `p3`, `p4`, `expected_3`,
`expected_4`.

In [ ]:
# ← your answer here: replace every ... with code
n_week = 163  # aftershocks expected next week, days 11 to 17 (Your turn 2)

p3 = (...).mean()       # the proportion of sample with a magnitude of 3 or more
p4 = (...).mean()       # the proportion of sample with a magnitude of 4 or more
expected_3 = ... * ...  # number of tries times probability: n_week times p3
expected_4 = ... * ...  # the same with p4

print((sample["mag"] >= 3).sum(), "of", len(sample), "are magnitude 3 or more: a proportion of", round(p3, 3))
print((sample["mag"] >= 4).sum(), "of", len(sample), "are magnitude 4 or more: a proportion of", round(p4, 3))
print("expected next week:", round(expected_3), "of magnitude 3 or more,", round(expected_4, 1), "of magnitude 4 or more")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(p3 - (sample["mag"] >= 3).mean()) < 1e-9 and 0.30 < p3 < 0.37, "p3 is the proportion of the sample with magnitude 3 or more: the mean of a mask on sample, about a third"
assert abs(p4 - (sample["mag"] >= 4).mean()) < 1e-9 and 0.01 < p4 < 0.03, "p4 is the proportion of the sample with magnitude 4 or more, about 0.02"
assert abs(expected_3 - n_week * p3) < 1e-9 and abs(expected_4 - n_week * p4) < 1e-9, "an expected number is the number of tries, n_week, times the probability"
print(f"✓ Your turn 3 — a probability of {round(p3, 3)} for magnitude 3 or more and {round(p4, 3)} for 4 or more: expect {round(expected_3)} and {round(expected_4, 1)} next week")

**A third of the aftershocks are magnitude 3 or more; about one in fifty is magnitude 4 or
more** (13 of 637). So of next week's 163, expect about 54 of
magnitude 3 or more and about 3.3 of magnitude 4 or more.

## 4. What is the probability of a magnitude 4 or more next week?

Expect about 3.3 aftershocks of magnitude 4 or more. But that is an average. The week
that comes will hold a whole number of them: 0, 1, 2, 3 or more. **What is the probability that
it holds at least one?**

**The plan: let the computer live through next week many times, and count the weeks with at
least one.** The proportion of such weeks is the probability. This is a **simulation**, and it
needs random numbers.

### Random numbers in Python

**A random variable is a number that chance decides**: the face a die shows, the magnitude of
the next aftershock. You cannot say what its next value will be. You can say how often each
value comes up over many tries.

**Python draws random values with two tools:**

- `rng = np.random.default_rng(seed)` makes a random-number generator. A computer's random numbers come from a recipe, and the seed is where the recipe starts, so the same seed gives the same draws every time.
- `rng.choice(values, size)` draws `size` of the values at random. Every value has the same
  chance at every draw, so a value can come up more than once.

The cell rolls a die: the values are the six faces.

In [ ]:
rng = np.random.default_rng(1)                    # a random-number generator, started from the seed 1
die = [1, 2, 3, 4, 5, 6]                          # the six faces

rolls = rng.choice(die, 10)                       # roll the die 10 times
print("ten rolls:     ", rolls)

more_rolls = rng.choice(die, 10)                  # roll 10 more times: the numbers are new
print("ten more rolls:", more_rolls)

many_rolls = rng.choice(die, 6000)                # roll 6,000 times
sixes = (many_rolls == 6).sum()                   # how many of the rolls are a six
print("sixes in 6,000 rolls:", sixes, "  a proportion of", round(sixes / 6000, 3))

**Every call of `rng.choice` gives new values**, as every throw of a real die does. Nobody can
say what one roll will be. **Over many rolls the proportion settles**: about 1,000 of the 6,000
rolls are a six, a proportion near 1/6 = 0.167, the probability of a six.

### One possible week

**Now replace the die with the earthquakes.** The values are the magnitudes in `sample`, and one
draw is the magnitude of one of next week's aftershocks. **One possible week is 163 draws.**

In [ ]:
n_week = 163                                      # aftershocks expected next week, days 11 to 17 (Your turn 2)

one_week = rng.choice(sample["mag"], n_week)      # one possible week: n_week magnitudes drawn from the sample
print("its first five magnitudes:", one_week[:5])
print("one possible week:", (one_week >= 4).sum(), "of magnitude 4 or more")

another_week = rng.choice(sample["mag"], n_week)  # draw again: another possible week
print("another possible week:", (another_week >= 4).sum(), "of magnitude 4 or more")

**Each draw gives a different week**, with its own number of aftershocks of magnitude 4 or more.
One week says little. Many weeks give a probability.

### ✏️ Your turn 4

**In what proportion of 2,000 possible weeks is there at least one aftershock of magnitude 4
or more?**

**Why:** that proportion is the probability the forecast needs. One possible week cannot give
it; 2,000 can, as 6,000 rolls gave the probability of a six.

1. **Draw 2,000 possible weeks** with a loop. Each time, draw one week as the cell above did,
   count its magnitudes of 4 or more, and keep that count in the list `larger`. After the loop,
   `np.array` turns the list into an array.
2. **`prob_sim`**: the proportion of the 2,000 counts that are 1 or more. This is the
   probability.
3. **Two more numbers describe the 2,000 counts**:
   - `mean_larger`: the **mean**, `.mean()`, the average count in a week.
   - `std_larger`: the **standard deviation**, `.std()`. How far a value usually is from the average. A small
     one means the weeks are much alike; a large one means they differ a lot.

**Use these names**, because the self-check looks for them: `larger`, `prob_sim`, `mean_larger`,
`std_larger`.

In [ ]:
# ← your answer here: replace every ... with code
n_week = 163  # aftershocks expected next week (Your turn 2); sample holds the aftershocks already recorded

rng = np.random.default_rng(88)  # a new generator, started from the seed 88
larger = []                      # one count for each possible week
for run in range(2000):
    week = rng.choice(..., ...)  # one possible week: n_week magnitudes drawn from the sample
    larger.append(...)           # how many of the week's magnitudes are 4 or more
larger = np.array(larger)

prob_sim = (...).mean()  # the proportion of the weeks where larger is 1 or more
mean_larger = ...        # the mean of larger
std_larger = ...         # the standard deviation of larger

print("the first ten weeks:", larger[:10])
print("at least one in", round(prob_sim, 3), "of the weeks")
print("mean", round(mean_larger, 1), "  standard deviation", round(std_larger, 1))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert len(larger) == 2000 and larger.max() < 40, "larger holds one number for each of the 2000 weeks: how many of that week's aftershocks are magnitude 4 or more"
assert 3.08 < larger.mean() < 3.57 and 1.53 < larger.std() < 2.08, "each week is n_week magnitudes drawn from the sample's mag column, and larger keeps the COUNT of those that are 4 or more: about 3.3 on average"
assert abs(prob_sim - (larger >= 1).mean()) < 1e-9, "prob_sim is the proportion of weeks where larger is 1 or more: the mean of a mask"
assert abs(mean_larger - larger.mean()) < 1e-9 and abs(std_larger - larger.std()) < 1e-9, "mean_larger is larger.mean() and std_larger is larger.std()"
print(f"✓ Your turn 4 — at least one in {round(100 * prob_sim)}% of 2000 weeks; mean {round(mean_larger, 1)}, standard deviation {round(std_larger, 1)}")

In [ ]:
plt.figure(figsize=(7, 3.6))
plt.hist(larger, bins=np.arange(0, larger.max() + 2), align="left")       # one bar for each whole number
plt.axvline(mean_larger, color="firebrick")                                 # the mean
plt.axvline(mean_larger - std_larger, color="firebrick", ls="--")           # one standard deviation below it
plt.axvline(mean_larger + std_larger, color="firebrick", ls="--")           # and one above it
plt.xlabel("aftershocks of magnitude 4 or more in the week")
plt.ylabel("number of weeks")
plt.title("2,000 possible weeks (n = 2,000)")
plt.show()

inside = (larger > mean_larger - std_larger) & (larger < mean_larger + std_larger)
print("weeks within one standard deviation of the mean:", round(inside.mean(), 2))
for tries in [10, 100, 1000, 2000]:
    print("after", tries, "weeks, the proportion with at least one:", round((larger[:tries] >= 1).mean(), 2))

### What the 2,000 weeks say

The histogram is a **distribution**: it shows how often each outcome came up. Your three numbers
describe it.

- **The probability of at least one is 0.962.** That is every bar but the first,
  the weeks with none.
- **The mean is 3.3**, the solid line. It is the expected number of Your turn 3, found
  a second way.
- **The standard deviation is 1.8.** The dashed lines stand one standard deviation
  below and above the mean, at 1.5 and 5.1, and 72% of the weeks
  fall between them. So "about 3.3" means "usually between 2 and
  5".

**Why so many weeks?** The last lines of the printout answer. After 10 weeks the proportion
reads 1; by 100 it is near 0.96, and it stays there. A
probability is the value the proportion settles on. Finding one by many random tries is called
the **Monte Carlo** method. Let the computer live through the same situation thousands of times, each time with fresh random draws, and count how often a thing happens.

### The rules of probability behind it

**The same probability can be worked out with three rules and no simulation.**

**Rule 1: not.** The probability that a thing does *not* happen is 1 minus the probability that
it does. One aftershock is magnitude 4 or more with probability 0.02, so it is smaller
with probability 1 − 0.02 = 0.98.

**Rule 2: and.** If two things are independent, the probability that *both* happen is the product
of their two probabilities. A coin lands tails with probability 0.5, so two coins both land tails
with probability 0.5 × 0.5 = 0.25: one time in four.

**Independent.** Knowing how one turned out tells you nothing about the next. Two coins are independent. Two aftershocks nearly are:
the size of one says almost nothing about the size of the next.

Two aftershocks are both smaller than magnitude 4 with probability
0.98 × 0.98 = 0.96. Three are, with probability
0.98 × 0.98 × 0.98 = 0.94. For any number of
aftershocks, multiply 0.98 by itself that many times. Python writes this with `**`:
`0.98 ** 3` is 0.98 × 0.98 × 0.98.

**Rule 3: at least one.** "At least one is magnitude 4 or more" is the opposite of "all of them
are smaller". So by rule 1 its probability is 1 minus the probability from rule 2.

The cell uses the three rules for every number of aftershocks from 1 to 300, and draws both
probabilities.

In [ ]:
p4 = 13 / 637                                     # the probability that one aftershock is magnitude 4 or more (Your turn 3)

counts = np.arange(1, 301)                        # 1, 2, 3, ... 300 aftershocks
all_smaller = (1 - p4) ** counts                  # rules 1 and 2: every one of them is below magnitude 4
at_least_one = 1 - all_smaller                    # rule 3: at least one is magnitude 4 or more

plt.figure(figsize=(7, 3.6))
plt.plot(counts, all_smaller, color="gray", label="all are below magnitude 4")
plt.plot(counts, at_least_one, color="firebrick", label="at least one is magnitude 4 or more")
plt.xlabel("number of aftershocks")
plt.ylabel("probability")
plt.title("The more aftershocks, the more likely a large one")
plt.legend()
plt.show()

for count in [1, 10, 50, 100]:
    prob = 1 - (1 - p4) ** count                  # the three rules in one line
    print("among", count, "aftershocks: at least one with probability", round(prob, 2))

**Each aftershock is unlikely to be large, but there are many of them.** One is magnitude 4 or
more with probability 0.02. Among 10 there is at least one with probability
0.19, among 50 with 0.64, among 100 with 0.87. The two curves
always add up to 1, by rule 1: either all are smaller, or at least one is not.

### ✏️ Your turn 5

**What is the probability of at least one aftershock of magnitude 4 or more in the whole week,
and tomorrow alone?**

**Why:** the rules give in one line what the simulation needed 2,000 weeks for, and they work
for any number of aftershocks. Use them first for the week's `n_week` aftershocks, to compare
with your `prob_sim`, then for tomorrow's `n_tomorrow`.

**Use these names**, because the self-check looks for them: `p_small`, `prob_none`, `prob_week`,
`prob_tomorrow`.

In [ ]:
# ← your answer here: replace every ... with code
n_week = 163     # aftershocks expected in the week, days 11 to 17
n_tomorrow = 29  # aftershocks expected tomorrow, day 11
p4 = 13 / 637    # the probability that one aftershock is magnitude 4 or more (Your turn 3)

p_small = 1 - ...               # rule 1: one aftershock is below magnitude 4
prob_none = ... ** ...          # rule 2: all n_week aftershocks of the week are below magnitude 4
prob_week = 1 - ...             # rule 3: at least one in the week is magnitude 4 or more
prob_tomorrow = 1 - ... ** ...  # rules 2 and 3 in one line, for tomorrow's n_tomorrow aftershocks

print("the week: none with probability", round(prob_none, 3), "  at least one with probability", round(prob_week, 3))
print("tomorrow: at least one with probability", round(prob_tomorrow, 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(p_small - (1 - p4)) < 1e-9 and abs(prob_none - (1 - p4) ** n_week) < 1e-9, "p_small is 1 minus p4, and prob_none is p_small raised to the power n_week"
assert abs(prob_week - (1 - (1 - p4) ** n_week)) < 1e-9, "prob_week is 1 minus prob_none"
assert abs(prob_tomorrow - (1 - (1 - p4) ** n_tomorrow)) < 1e-9, "prob_tomorrow is 1 minus p_small raised to the power n_tomorrow"
print(f"✓ Your turn 5 — at least one of magnitude 4 or more: {round(100 * prob_tomorrow)}% tomorrow, {round(100 * prob_week)}% in the week")

**The rules give 97% for the week; the simulation gave
96%.** The two agree because they assume the same things: 163
aftershocks come, each is magnitude 4 or more with probability 0.02, and the sizes are
independent. Tomorrow alone is close to an even chance, 45%; over the week
it is nearly certain.

**When to use which.** The rules are quicker. The simulation also gives the mean and the
standard deviation, and it answers questions these rules do not, such as the probability of two
or more (homework part 3).

**Your forecast for days 11 to 17:**

- about 163 aftershocks;
- about 54 of them magnitude 3 or more;
- a 97% probability of at least one of magnitude 4 or more, and usually
  2 to 5 of them.

Now look at what the Earth did.

In [ ]:
came = quakes[(quakes["days"] >= 10) & (quakes["days"] < 17)]   # days 11 to 17: from 10 days to 17 days after the mainshock
came_3 = (came["mag"] >= 3).sum()
came_4 = (came["mag"] >= 4).sum()

print("aftershocks in days 11 to 17:", len(came), "   expected:", 163)
print("of magnitude 3 or more:", came_3, "   expected:", 54)
print("of magnitude 4 or more:", came_4, "   expected:", 3.3)
print("the largest:", came["mag"].max())

past_days = np.arange(1, 11)                    # the day numbers 1 to 10; daily holds the count of each
coming_days = np.arange(11, 18)                 # the day numbers 11 to 17
came_daily, edges = np.histogram(came["days"], bins=np.arange(10, 18))   # what came on each of days 11 to 17

plt.figure(figsize=(7, 3.6))
plt.scatter(past_days, daily, label="the count of each day, days 1 to 10")
plt.plot(past_days, 320 / past_days, color="firebrick", label="the model, K / day with K = 320")
plt.plot(coming_days, 320 / coming_days, color="firebrick", ls="--", label="the forecast, days 11 to 17")
plt.scatter(coming_days, came_daily, color="darkorange", label="what came")
plt.yscale("log")
plt.xlabel("day after the mainshock")
plt.ylabel("aftershocks in the day")
plt.title(f"The forecast and what came (n = {len(came)} aftershocks in days 11 to 17)")
plt.legend()
plt.show()

**The forecast held.** 169 aftershocks came where you expected 163; 62 of
magnitude 3 or more where you expected 54; 3 of magnitude 4 or more where you
expected 3.3, give or take 1.8.

**The official forecast was close to yours.** The US Geological Survey published its own on
15 July 2019, for nearly the same week: 23 to 52 aftershocks of magnitude 3 or more, and a
probability above 99% of at least one. Your 54 is just above that range (the Survey does
not count over exactly this box). Its forecast has the same two ingredients as yours: how fast
the count falls, and what proportion of aftershocks are large.

**Two limits of your forecast:**

- **It is rough.** The curve runs above some dots and below others. The Survey renews its
  forecast as the sequence goes on.
- **It cannot reach sizes the sample does not hold.** The Survey also gave a 30% probability of
  a magnitude 5 or more. Your sample holds none (its largest is 4.9), so its
  proportion is 0, and that cannot be right. The transformations week shows how to reach sizes
  too rare to be in the sample.

One aftershock was larger than all of these: a magnitude 5.5, about
11 months after the mainshock.

## The question, answered

Ten days after the Ridgecrest mainshock about 160 more aftershocks were due within
a week and one in fifty aftershocks was magnitude 4 or more, so at least one more of that size
was nearly certain, and 3 came.

## Week 5 summary

### The science

**A big earthquake has just struck. What will the next week bring?**  
A large earthquake raises the probability of more earthquakes around it, and that extra rate dies away roughly as one over the time since it. A probability is a proportion of many tries: of what has happened, or of futures you simulate. A forecast is a count and a proportion turned into an expected number and a probability of at least one, and the week that comes can still differ from it.

- **Probability.** The proportion of many tries in which a thing happens.
- **Omori's law.** The number of aftershocks in a day falls roughly as one over the time since the mainshock: twice as late, about half as many.
- **Expected number.** How many to expect on average: the number of tries times the probability of each.
- **Monte Carlo.** Let the computer live through the same situation thousands of times, each time with fresh random draws, and count how often a thing happens.
- **Standard deviation.** How far a value usually is from the average.
- **Independent.** Knowing how one turned out tells you nothing about the next.
- **Aftershock sequence.** A large earthquake changes the stress in the rock around it, and parts of the same fault and faults near it that were already close to failing then fail too — most within hours and days, fewer and fewer over the months that follow.

### The code

| Function | What it does |
|---|---|
| **Python** | |
| `len(table)` | how many rows |
| `for x in things:` | do the same thing once for each item |
| `range(n)` | the numbers 0 to n-1, to count with |
| `list.append(x)` | add one item to the end of a list |
| `round(x, 1)` | trim a long decimal — 9.1 rather than sixteen digits |
| **NumPy** | |
| `array.std()` | the standard deviation: how far a value usually is from the average |
| `np.random.default_rng(seed)` | a random-number generator you can reproduce — the same seed gives the same draws |
| `rng.choice(values, n)` | n of the values drawn at random; the same one may be drawn more than once |
| `np.array(list)` | the container that does arithmetic to every number at once |
| **pandas** | |
| `mask.mean()` | the proportion of a mask that is True, in one line |
| `series.mean()` | the average value |
| `table[mask]` | keep only the rows where the mask is True |
| **Matplotlib** | |
| `plt.plot(x, y)` | join points with a line — here, the model K / day drawn through the daily counts |

## Homework

Three parts on the Bay Area's own earthquake. At 5:04 in the afternoon of 17 October 1989 the
magnitude 6.9 Loma Prieta earthquake struck the Santa Cruz Mountains, about 100 km south of
campus. It is the earthquake that interrupted the 1989 World Series and broke a section of the
Bay Bridge. `loma` holds every event of magnitude 2.5 or more within 60 km of it, from the
start of 1989 to a year after the mainshock: 644 rows, with the same `days` column,
counted from its own mainshock. The first cell below shows the sequence.

The three parts make and test a forecast for Loma Prieta, as class did for Ridgecrest: the model
K / day and the three rules in part 1, the rules again in part 2, the simulation loop in part 3.
If you have restarted the kernel, run the setup cell at the top; nothing else from class is
needed.

In [ ]:
plt.figure(figsize=(7, 3.6))
plt.scatter(loma["days"], loma["mag"], s=5)
plt.xlim(-60, 60)
plt.xlabel("days since the magnitude 6.9")
plt.ylabel("magnitude")
plt.title("Loma Prieta: every event from two months before to two months after")
plt.show()

### ✏️ Your turn 6

**Ten days after Loma Prieta, what was the probability of an aftershock of magnitude 4 or more in the
next week?**

**Why:** it is the forecast of class, made on your own for another earthquake, from its first
ten days alone.

1. **The first ten days.** `loma_ten` holds their aftershocks; the cell counts them day by day.
2. **Find K.** Change `K_loma` until the red curve passes through the dots from day 2 on. This
   fit is rougher than Ridgecrest's, because there are fewer aftershocks.
3. **Forecast the count.** `n_week_loma` is the model added up over days 11 to
   17, rounded to a whole number.
4. **The probability.** `sample_loma` holds the aftershocks from 1 day to 10 days after the
   mainshock, as in class and for the same two reasons. `p_loma` is its proportion of magnitude 4 or more. `prob_loma` is
   the probability of at least one, by the three rules.

The last lines print what came and draw your forecast against it.

**Use these names**, because the self-check looks for them: `loma_ten`, `K_loma`,
`n_week_loma`, `sample_loma`, `p_loma`, `prob_loma`.

In [ ]:
# ← your answer here: replace every ... with code
# Step 1: the first ten days
loma_ten = loma[(...) & (...)]                                             # the first 10 days after the mainshock: days above 0, and days below 10
loma_daily, edges = np.histogram(loma_ten["days"], bins=np.arange(0, 11))  # the count of each day
past_days = np.arange(1, 11)                                               # the day numbers 1 to 10

# Step 2: find K
K_loma = ...  # try 20 first, then change it until the curve passes through the dots
plt.figure(figsize=(7, 3.6))
plt.scatter(past_days, loma_daily, label="the count of each day")
# in the next line, write the model on the past days: K_loma over past_days
plt.plot(past_days, ... / ..., color="firebrick", label="the model, K / day")
plt.yscale("log")
plt.xlabel("day after the mainshock")
plt.ylabel("aftershocks in the day")
plt.title(f"Loma Prieta: Omori's law with K = {K_loma} (n = {len(past_days)} days)")
plt.legend()
plt.show()

# Step 3: forecast the count
coming_days = np.arange(11, 18)         # the day numbers 11 to 17: the week to forecast
n_week_loma = round((... / ...).sum())  # the model on the coming days, added up and rounded

# Step 4: the probability
sample_loma = loma[(...) & (...)]  # from 1 day to 10 days after the mainshock: days at least 1, and days below 10
p_loma = (...).mean()              # the proportion of sample_loma with a magnitude of 4 or more
prob_loma = 1 - (1 - ...) ** ...   # at least one among the n_week_loma aftershocks, by the three rules

# What came (nothing to change below)
came_loma = loma[(loma["days"] >= 10) & (loma["days"] < 17)]  # days 11 to 17: from 10 days to 17 days after the mainshock
print("K =", round(K_loma), "  ", n_week_loma, "expected in days 11 to 17")
print((sample_loma["mag"] >= 4).sum(), "of", len(sample_loma), "in the sample are magnitude 4 or more: a proportion of", round(p_loma, 2))
print(f"the probability of at least one of magnitude 4 or more: {round(100 * prob_loma)}%")
print("came:", len(came_loma), "aftershocks,", (came_loma["mag"] >= 4).sum(), "of magnitude 4 or more")

came_daily, edges = np.histogram(came_loma["days"], bins=np.arange(10, 18))
plt.figure(figsize=(7, 3.6))
plt.scatter(past_days, loma_daily, label="the count of each day, days 1 to 10")
plt.plot(past_days, K_loma / past_days, color="firebrick", label="the model, K / day")
plt.plot(coming_days, K_loma / coming_days, color="firebrick", ls="--", label="your forecast, days 11 to 17")
plt.scatter(coming_days, came_daily, color="darkorange", label="what came")
plt.yscale("log")
plt.xlabel("day after the mainshock")
plt.ylabel("aftershocks in the day")
plt.title("Loma Prieta: the forecast and what came")
plt.legend()
plt.show()

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert (loma_ten["days"] > 0).all() and (loma_ten["days"] < 10).all() and len(loma_ten) > 300, "loma_ten keeps the rows of loma with days above 0 and below 10"
assert 25 <= K_loma <= 70, "choose K_loma so that the curve passes through the dots from day 2 on: a value between about 25 and 70"
assert n_week_loma == round((K_loma / coming_days).sum()), "n_week_loma is the model on the COMING days, K_loma / coming_days, added up and rounded"
assert len(sample_loma) == ((loma["days"] >= 1) & (loma["days"] < 10)).sum() and (sample_loma["days"] >= 1).all() and abs(p_loma - (sample_loma["mag"] >= 4).mean()) < 1e-9, "sample_loma keeps every row of loma from 1 day to 10 days after the mainshock, and p_loma is its proportion of magnitude 4 or more"
assert abs(prob_loma - (1 - (1 - p_loma) ** n_week_loma)) < 1e-9, "prob_loma is 1 minus the probability that all n_week_loma are below magnitude 4"
print(f"✓ Homework 1 — {n_week_loma} aftershocks expected, a proportion of {round(p_loma, 2)} at magnitude 4 or more, so a {round(100 * prob_loma)}% probability of at least one")

**What the numbers mean.** Compare your forecast with the last line of your printout. Did the count come close, and did the
likely thing happen? Then think about what the proportion rests on: how many earthquakes of magnitude 4
or more are in your sample? Print them with `sample_loma[sample_loma["mag"] >= 4]["mag"]` if you like.
They are listed to one decimal. What would happen to your proportion if the question had
been "more than 4" and not "4 or more"?

### ✏️ Your turn 7

**Whose proportion should the forecast use: Loma Prieta's own, or Ridgecrest's?**

**Why:** ten days into a sequence a forecaster has few large aftershocks to measure a proportion
from: here 12 out of 107. Ridgecrest offers a second measurement, from more
aftershocks: 13 out of 637. The two give different forecasts.

1. `p_ridge`: Ridgecrest's proportion of magnitude 4 or more, from its sample `sample_ridge`.
2. `prob_own` and `prob_borrowed`: the probability of at least one, with each proportion.
3. `expected_own` and `expected_borrowed`: the expected number, with each proportion.

Part 1's two results are written in the cell, so this part stands on its own.

**Use these names**, because the self-check looks for them: `p_ridge`, `prob_own`,
`prob_borrowed`, `expected_own`, `expected_borrowed`.

In [ ]:
# ← your answer here: replace every ... with code
n_week_loma = 24                                                      # from part 1: aftershocks expected at Loma Prieta in days 11 to 17
p_loma = 12 / 107                                                     # from part 1: Loma Prieta's own proportion of magnitude 4 or more
sample_ridge = quakes[(quakes["days"] >= 1) & (quakes["days"] < 10)]  # Ridgecrest's sample from class: from 1 day to 10 days after its mainshock

p_ridge = (...).mean()                # the proportion of sample_ridge with a magnitude of 4 or more
prob_own = 1 - (1 - ...) ** ...       # at least one among the n_week_loma, with p_loma
prob_borrowed = 1 - (1 - ...) ** ...  # the same with p_ridge
expected_own = ... * ...              # number of tries times probability: n_week_loma times p_loma
expected_borrowed = ... * ...         # the same with p_ridge

print(f"Loma Prieta's own proportion, {round(p_loma, 3)}: a {round(100 * prob_own)}% probability, {round(expected_own, 1)} expected")
print(f"Ridgecrest's proportion, {round(p_ridge, 3)}: a {round(100 * prob_borrowed)}% probability, {round(expected_borrowed, 1)} expected")

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert abs(p_ridge - (sample_ridge["mag"] >= 4).mean()) < 1e-9, "p_ridge is the proportion of sample_ridge, Ridgecrest's aftershocks, that is magnitude 4 or more"
assert abs(prob_own - (1 - (1 - p_loma) ** n_week_loma)) < 1e-9 and abs(prob_borrowed - (1 - (1 - p_ridge) ** n_week_loma)) < 1e-9, "each probability is 1 minus the probability that all n_week_loma are below magnitude 4: p_loma in one, p_ridge in the other"
assert abs(expected_own - n_week_loma * p_loma) < 1e-9 and abs(expected_borrowed - n_week_loma * p_ridge) < 1e-9, "an expected number is the number of tries, n_week_loma, times the proportion"
print(f"✓ Homework 2 — a {round(100 * prob_own)}% probability with Loma Prieta's own proportion, {round(100 * prob_borrowed)}% with Ridgecrest's")

**What the numbers mean.** Your printout holds two forecasts made with the same method. How far apart are they, and which
would you have published on day 10? Think about what each proportion rests on: Loma Prieta's own is
about the right earthquake, but how many aftershocks of magnitude 4 or more stand behind it?
How many stand behind Ridgecrest's? It is also a different fault, thirty years later, recorded by
different instruments. What extra data would tell you which proportion was nearer the truth?

### ✏️ Your turn 8

**2 aftershocks of magnitude 4 or more came. Which forecast does that favour, and what
did the year that followed say?**

**Why:** the two proportions of part 2 disagree, and what came can test them: under which
forecast is "2 or more" an ordinary week? The three rules do not answer
"2 or more". A simulation does.

1. **Simulate the week 2,000 times with each sample**, as Your turn 4 did with one. This time
   the loop is yours to write: the generator, the `for` line and what each `append` keeps. `own`
   keeps the count of magnitude 4 or more when the 24 magnitudes are drawn from
   `sample_loma`; `borrowed`, when they are drawn from `sample_ridge`.
2. **`prop_own` and `prop_borrowed`**: the proportion of the weeks with 2 or more, for
   each.
3. **The later earthquakes.** `later` holds the earthquakes more than 17 days
   after the mainshock, which a forecaster on day 10 could not see. `p_later` is their proportion of magnitude 4 or more.

**Use these names**, because the self-check looks for them: `sample_loma`, `own`, `borrowed`,
`prop_own`, `prop_borrowed`, `later`, `p_later`.

In [ ]:
# ← your answer here: replace every ... with code
n_week_loma = 24                                                      # from part 1: aftershocks expected at Loma Prieta in days 11 to 17
sample_loma = loma[(...) & (...)]                                     # Loma Prieta's sample again: from 1 day to 10 days after its mainshock
sample_ridge = quakes[(quakes["days"] >= 1) & (quakes["days"] < 10)]  # Ridgecrest's sample from class: from 1 day to 10 days after its mainshock

rng = ...                 # a random-number generator, started from the seed 1989
own = []
borrowed = []
for run in ...:           # 2,000 times
    own.append(...)       # draw n_week_loma magnitudes from sample_loma, and count those of 4 or more
    borrowed.append(...)  # the same, drawn from sample_ridge
own = np.array(own)
borrowed = np.array(borrowed)

prop_own = (...).mean()       # the proportion of the own weeks with 2 or more
prop_borrowed = (...).mean()  # the proportion of the borrowed weeks with 2 or more
later = loma[...]             # the rows more than 17 days after the mainshock: days at least 17
p_later = (...).mean()        # their proportion of magnitude 4 or more

print(f"weeks with 2 or more: {round(100 * prop_own)}% with Loma Prieta's own sample, {round(100 * prop_borrowed)}% with Ridgecrest's")
print((later["mag"] >= 4).sum(), "of", len(later), "earthquakes after day 17 were magnitude 4 or more: a proportion of", round(p_later, 3))

In [ ]:
# Self-check: run this after your answer. You do not need to read it — it only says whether something is off.
assert len(own) == 2000 and len(borrowed) == 2000, "own and borrowed each hold one number for every one of the 2000 weeks"
assert len(sample_loma) == ((loma["days"] >= 1) & (loma["days"] < 10)).sum() and (sample_loma["days"] >= 1).all(), "sample_loma keeps every row of loma from 1 day to 10 days after the mainshock, as in part 1"
assert 2.48 < own.mean() < 2.9 and 0.4 < borrowed.mean() < 0.58, "own draws n_week_loma sizes from sample_loma and keeps the COUNT of magnitude 4 or more, about 2.7 on average; borrowed draws them from sample_ridge, about 0.5"
assert abs(prop_own - (own >= 2).mean()) < 1e-9 and abs(prop_borrowed - (borrowed >= 2).mean()) < 1e-9, "each proportion is the proportion of weeks with 2 or more: the mean of a mask on own, and on borrowed"
assert (later["days"] >= 17).all() and len(later) == (loma["days"] >= 17).sum() and abs(p_later - (later["mag"] >= 4).mean()) < 1e-9, "later keeps the rows of loma from day 17 on, and p_later is its proportion of magnitude 4 or more"
print(f"✓ Homework 3 — 2 or more in {round(100 * prop_own)}% of the own weeks and {round(100 * prop_borrowed)}% of the borrowed ones; the later proportion was {round(p_later, 3)}")

**What the numbers mean.** Look at your two proportions. Under which forecast was "2 or more" an ordinary week, and
under which an unusual one? Does an unusual week prove a forecast wrong? Then compare the proportion
you measured from the later earthquakes with the two proportions of part 2. Where does it fall? Loma Prieta's own proportion rested on 12 earthquakes; think about how far a proportion
resting on so few could be from the truth.